# 4 · Evaluate (GPU job) + view results

Scores base vs SFT vs SFT+GRPO with the **eval judge** (different from the train judge). Produces: pairwise win-rate (headline), absolute rubric means, length, ROUGE. Uploads metrics + chart to `{results_s3}`, then downloads and shows them.

**How to read it:** the headline is `SFT+GRPO_vs_SFT` win-rate. Check `len_words` first — if SFT+GRPO only won by getting longer, that's verbosity hacking, not quality.

In [ ]:
import json
from sagemaker.core import image_uris
from sagemaker.core.helper.session_helper import Session
from sagemaker.train.model_trainer import ModelTrainer
from sagemaker.train.configs import SourceCode, Compute, StoppingCondition

sess = Session()
cfg = json.load(open('summ_config.json'))
region = cfg['region']
INSTANCE = cfg['gpu_instance']
image_uri = cfg['gpu_image_uri'] or image_uris.retrieve(
    framework='pytorch', region=region, version=cfg['pt_version'],
    instance_type=INSTANCE, image_scope='training')

env = {
    'BASE_MODEL': cfg['base_model'], 'DATA_S3': cfg['data_s3'],
    'SFT_MODEL_S3': cfg['sft_model_s3'], 'GRPO_MODEL_S3': cfg['grpo_model_s3'],
    'OUTPUT_RESULTS_S3': cfg['results_s3'],
    'SEQ_LEN': str(cfg['seq_len']), 'SEED': str(cfg['seed']), 'N_PAIRWISE': str(cfg['n_pairwise']),
    'JUDGE_BACKEND': cfg['judge_backend'], 'EVAL_JUDGE_MODEL': cfg['eval_judge_model'],
    'TRAIN_JUDGE_MODEL': cfg['train_judge_model'],
    'JUDGE_REGION': cfg.get('judge_region', '') or region, 'JUDGE_MAX_CONCURRENCY': str(cfg['judge_max_concurrency']),
    'REWARD_WEIGHTS': cfg['reward_weights'], 'EVAL_BASE': str(cfg['eval_base']),
    'EVAL_MAX_NEW_TOKENS': str(cfg['eval_max_new_tokens']), 'EVAL_BATCH_SIZE': str(cfg['eval_batch_size']),
    'HF_HOME': '/opt/ml/hf_home', 'HF_HUB_ENABLE_HF_TRANSFER': '1', 'TOKENIZERS_PARALLELISM': 'false',
}
if cfg.get('hf_token'):
    env['HF_TOKEN'] = cfg['hf_token']; env['HUGGING_FACE_HUB_TOKEN'] = cfg['hf_token']
source_code = SourceCode(source_dir='../src', requirements='requirements.txt', entry_script='evaluate.sh')
ModelTrainer(sagemaker_session=sess, role=cfg['role'], training_image=image_uri,
    source_code=source_code, compute=Compute(instance_type=INSTANCE, instance_count=1, keep_alive_period_in_seconds=cfg.get('keep_alive', 1800)),
    environment=env, base_job_name='summ-eval',
    stopping_condition=StoppingCondition(max_runtime_in_seconds=12*3600)).train(wait=True)
print('results ->', cfg['results_s3'])

### Download + display results

In [ ]:
import boto3, json, os
from urllib.parse import urlparse
u = urlparse(cfg['results_s3']); bucket, prefix = u.netloc, u.path.strip('/')
s3 = boto3.client('s3', region_name=region)
os.makedirs('results', exist_ok=True)
for key in ('metrics.json', 'metrics.csv', 'comparison.png', 'samples.json'):
    try:
        s3.download_file(bucket, f'{prefix}/{key}', f'results/{key}')
    except Exception as e:
        print('skip', key, e)
m = json.load(open('results/metrics.json'))
import pandas as pd
print('train judge:', m['train_judge'])
print('eval  judge:', m['eval_judge'], '(must differ for an honest win-rate)')
display(pd.DataFrame(m['per_model']).set_index('model').round(4))
print('\nPairwise win-rates:')
for name, wr in m['pairwise'].items():
    print(f"  {name}: win={wr['win']:.3f} tie={wr['tie']:.3f} loss={wr['loss']:.3f} (n={wr['n']})")

In [ ]:
from IPython.display import Image
Image('results/comparison.png')